# PageRank: Tiny Web Popularity Simulator

PageRank estimates which pages are important by watching how attention flows through links.

Larry Page and Sergey Brin used PageRank in the early Google search engine to treat links as signals of reputation. The same graph-ranking idea appears in recommendation systems, citation analysis, social networks, fraud detection, and influence modeling.

In this notebook, you will build it with small objects: pages, links, a web graph, and a runner that updates ranks over time.

<details>
<summary>Big idea</summary>

A page earns rank when other pages link to it. Links from already-important pages count more.

</details>

## 1. The Mental Model

A web is just a graph:

- **Page**: a node, like `Arcade` or `Library`
- **Link**: a directed edge from one page to another
- **Rank**: a score that represents importance
- **Round**: one pass where pages share rank through their links
- **Damping**: the chance a visitor keeps clicking instead of jumping anywhere

PageRank repeats one move: each page shares its rank across outgoing links, then every page gets a tiny random-jump boost.

<details>
<summary>Hint: why damping?</summary>

Real visitors do not click forever. Damping keeps the simulation stable and gives every page a small chance to be visited.

</details>

## 2. Build the Objects

Implementation plan:

1. `Page` stores a page name.
2. `WebGraph` stores pages and outgoing links.
3. `RankStep` records each simulation round.
4. `PageRankRunner` owns the rank update logic.

<details>
<summary>Implementation hint</summary>

A page with no outgoing links is called a dangling page. In this notebook, it shares its rank evenly with every page.

</details>

In [ ]:
from dataclasses import dataclass, field


### Define a page

- 

In [ ]:
@dataclass(frozen=True, order=True)
class Page:
    name: str

    def __str__(self) -> str:
        return self.name

    def __format__(self, spec: str) -> str:
        return format(self.name, spec)


### Define the web graph

- 

In [ ]:

@dataclass
class WebGraph:
    links: dict[Page, set[Page]] = field(default_factory=dict)

    def add_page(self, page: Page) -> None:
        self.links.setdefault(page, set())

    def link(self, source: Page, target: Page) -> None:
        self.add_page(source)
        self.add_page(target)
        self.links[source].add(target)

    def outgoing(self, page: Page) -> list[Page]:
        return sorted(self.links.get(page, set()))

    def pages(self) -> list[Page]:
        return sorted(self.links)

    def describe(self) -> str:
        rows = []
        for page in self.pages():
            targets = self.outgoing(page)
            label = ", ".join(str(target) for target in targets) or "no outgoing links"
            rows.append(f"{page:>10} -> {label}")
        return "\n".join(rows)

### Define the rank step

- 

In [ ]:
@dataclass
class RankStep:
    round_number: int
    ranks: dict[Page, float]
    note: str


### Define the page rank 

- 

In [ ]:

class PageRankRunner:
    def __init__(self, graph: WebGraph, damping: float = 0.85):
        if not 0 < damping < 1:
            raise ValueError("damping must be between 0 and 1.")
        self.graph = graph
        self.damping = damping

    def run(self, rounds: int = 12) -> tuple[dict[Page, float], list[RankStep]]:
        pages = self.graph.pages()
        if not pages:
            raise ValueError("PageRank needs at least one page.")

        page_count = len(pages)
        ranks = {page: 1 / page_count for page in pages}
        steps = [RankStep(0, ranks.copy(), "Start every page equal.")]

        for round_number in range(1, rounds + 1):
            next_ranks = {page: (1 - self.damping) / page_count for page in pages}
            dangling_rank = sum(ranks[page] for page in pages if not self.graph.outgoing(page))
            dangling_share = self.damping * dangling_rank / page_count

            for page in pages:
                next_ranks[page] += dangling_share

            for page in pages:
                targets = self.graph.outgoing(page)
                if not targets:
                    continue

                share = self.damping * ranks[page] / len(targets)
                for target in targets:
                    next_ranks[target] += share

            ranks = next_ranks
            steps.append(RankStep(round_number, ranks.copy(), f"After round {round_number}, rank moved through links."))

        return ranks, steps

    def top_pages(self, ranks: dict[Page, float], limit: int = 3) -> list[tuple[Page, float]]:
        return sorted(ranks.items(), key=lambda item: item[1], reverse=True)[:limit]

## 3. Create a Tiny Web

Now make a small web where pages link to each other. Links are one-way, just like real web links.

<details>
<summary>Hint: what should rank high?</summary>

A page usually ranks high when many pages point to it, especially if those pages also have strong rank.

</details>

In [4]:
arcade = Page("Arcade")
cafe = Page("Cafe")
library = Page("Library")
forum = Page("Forum")
zine = Page("Zine")
quiet_page = Page("Quiet Page")

web = WebGraph()
web.link(arcade, cafe)
web.link(arcade, library)
web.link(cafe, arcade)
web.link(cafe, library)
web.link(library, arcade)
web.link(forum, arcade)
web.link(forum, library)
web.link(zine, library)
web.link(zine, forum)
web.add_page(quiet_page)

print(web.describe())

    Arcade -> Cafe, Library
      Cafe -> Arcade, Library
     Forum -> Arcade, Library
   Library -> Arcade
Quiet Page -> no outgoing links
      Zine -> Forum, Library


## 4. Run PageRank

Start every page with the same rank. Then run several rounds until the scores settle down.

The runner returns two things:

- `ranks`: the final score for each page
- `steps`: snapshots for replaying the simulation

<details>
<summary>Quick check</summary>

`Library` should do well because several pages link to it.

</details>

In [5]:
runner = PageRankRunner(web)
ranks, steps = runner.run(rounds=12)

print("Final PageRank scores:")
for page, score in runner.top_pages(ranks, limit=len(ranks)):
    print(f"{page:>10}: {score:.3f}")

Final PageRank scores:
    Arcade: 0.394
   Library: 0.310
      Cafe: 0.196
     Forum: 0.042
Quiet Page: 0.029
      Zine: 0.029


## 5. Replay the Rounds

The replay shows how the ranking changes over time. Early rounds can jump around; later rounds usually stabilize.

<details>
<summary>Hint: what does stabilize mean?</summary>

It means each new round changes the scores less than the previous rounds did.

</details>

In [6]:
class RankReplay:
    def __init__(self, steps: list[RankStep]):
        self.steps = steps

    def show(self, rounds: list[int]) -> None:
        wanted = set(rounds)
        for step in self.steps:
            if step.round_number not in wanted:
                continue

            print(f"Round {step.round_number}: {step.note}")
            for page, score in sorted(step.ranks.items(), key=lambda item: item[1], reverse=True):
                print(f"  {page:>10}: {score:.3f}")
            print()


replay = RankReplay(steps)
replay.show(rounds=[0, 1, 2, 6, 12])

Round 0: Start every page equal.
      Arcade: 0.167
        Cafe: 0.167
       Forum: 0.167
     Library: 0.167
  Quiet Page: 0.167
        Zine: 0.167

Round 1: After round 1, rank moved through links.
      Arcade: 0.332
     Library: 0.332
        Cafe: 0.119
       Forum: 0.119
  Quiet Page: 0.049
        Zine: 0.049

Round 2: After round 2, rank moved through links.
      Arcade: 0.416
     Library: 0.295
        Cafe: 0.173
       Forum: 0.053
  Quiet Page: 0.032
        Zine: 0.032

Round 6: After round 6, rank moved through links.
      Arcade: 0.396
     Library: 0.310
        Cafe: 0.194
       Forum: 0.042
  Quiet Page: 0.029
        Zine: 0.029

Round 12: After round 12, rank moved through links.
      Arcade: 0.394
     Library: 0.310
        Cafe: 0.196
       Forum: 0.042
  Quiet Page: 0.029
        Zine: 0.029



## 6. Your Experiments

Try changing one thing at a time:

- Add more links to `Quiet Page`
- Remove one link to `Library`
- Change the damping value
- Add a new page that only links to `Forum`

<details>
<summary>Challenge</summary>

Predict the top page before you run the cell. Then compare your guess with the output.

</details>

In [7]:
experiment_web = WebGraph()
experiment_web.link(arcade, cafe)
experiment_web.link(arcade, library)
experiment_web.link(cafe, arcade)
experiment_web.link(cafe, library)
experiment_web.link(library, arcade)
experiment_web.link(forum, arcade)
experiment_web.link(forum, quiet_page)
experiment_web.link(zine, forum)
experiment_web.link(zine, quiet_page)
experiment_web.link(library, quiet_page)

experiment_runner = PageRankRunner(experiment_web, damping=0.85)
experiment_ranks, experiment_steps = experiment_runner.run(rounds=12)

print("Experiment top pages:")
for page, score in experiment_runner.top_pages(experiment_ranks, limit=3):
    print(f"{page:>10}: {score:.3f}")

Experiment top pages:
    Arcade: 0.257
   Library: 0.234
Quiet Page: 0.211


## Visual Trace + Rigor Studio

**Problem frame.** Rank nodes by stable importance under a random-surfer model.

**Interactive animation target.** Animate rank mass moving across links with damping and teleportation.

**Correctness handle.** Rank remains a probability distribution after every update.

**Complexity handle.** Power iteration is O(iterations * edges) on a sparse web graph.

**Failure mode to test.** Dangling nodes and disconnected components require damping or special handling.

**Studio task.** Add one link farm and explain how damping limits its influence.


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "courseware").exists():
        sys.path.insert(0, str(candidate))
        break

from courseware import AlgorithmPlayer, AlgorithmTrace, TraceStep, render_trace_table

# Convert the implementation above into snapshots:
# trace = AlgorithmTrace("Topic trace")
# trace.append("start", {"your_state": ...}, "What changed?", invariant="What remains true?")
# AlgorithmPlayer(trace, your_renderer).display()
print("Use AlgorithmTrace to turn this notebook's algorithm into a step-by-step visual player.")
